# Multi-Agent · Day 33 — **LangGraph Multi-Agent Supervisor**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2.5 hrs

Yesterday's crew ran a fixed chain. Today the route is **decided at runtime**: a supervisor inspects the state and picks the next specialist — over and over — until the work is done. This is the topology you'd actually take to a Barclays production review, because everything a regulator asks for (audit, fallback, human gate, resume) has a home in it.

Today:
1. **Typed message schemas** — the contract between agents.
2. Three specialists: **compliance-checker**, **data-retriever**, **report-generator**.
3. The **supervisor** — conditional routing on a real `StateGraph`.
4. **Failure + fallback routing** — a specialist dies, the request doesn't.
5. **Checkpointing + human gate**, and the supervisor's cost.

Keyless: the specialists are deterministic Python, and the supervisor routes on rules. Swap either for a Bedrock call and the graph is unchanged — that's the point of the seam.

## 1. Typed message schemas

Free-text between agents is where multi-agent systems rot. Pin the contract with Pydantic so a malformed handoff fails loudly, at the boundary.

In [1]:
from enum import StrEnum
from typing import Annotated, Literal, TypedDict
import operator
from pydantic import BaseModel, Field, ValidationError

class Specialist(StrEnum):
    COMPLIANCE = "compliance_checker"
    RETRIEVER  = "data_retriever"
    REPORTER   = "report_generator"
    DONE       = "__done__"

class AgentMessage(BaseModel):
    """The typed envelope every specialist returns to the supervisor."""
    sender: Specialist
    ok: bool = True
    summary: str = Field(min_length=3)
    data: dict = Field(default_factory=dict)
    error: str | None = None

class SupervisorState(TypedDict):
    request: str
    customer_id: str
    messages: Annotated[list[dict], operator.add]     # reducer: every hop appends
    next: str
    visited: Annotated[list[str], operator.add]
    failures: Annotated[list[str], operator.add]
    report: str

m = AgentMessage(sender=Specialist.RETRIEVER, summary="pulled 3 statements",
                 data={"accounts": 3})
print("valid  ->", m.sender, "|", m.summary, "|", m.data)

try:
    AgentMessage(sender="marketing_bot", summary="hi")
except ValidationError as exc:
    print("rejected ->", exc.errors()[0]["msg"][:70])

valid  -> data_retriever | pulled 3 statements | {'accounts': 3}
rejected -> Input should be 'compliance_checker', 'data_retriever', 'report_genera


☝ `Specialist` as a `StrEnum` does double duty: it's the message's `sender` field *and* the routing key the supervisor returns — so a typo like `"data_retreiver"` fails at validation instead of silently routing nowhere. `AgentMessage` is the contract; a specialist that can't fill it in is a specialist that's broken, and you find out at the handoff rather than in the final report.

The `Annotated[list, operator.add]` reducers (Day 02) are what make the supervisor loop safe — each hop *appends*, so no specialist can clobber another's trace.

## 2. The three specialists

Each is a node: read state, do work, return an `AgentMessage` plus a partial state update. Note the **access boundary** — only the retriever touches customer data, only the reporter writes `report`.

In [2]:
CUSTOMERS = {
    "C-4417": {"name": "ACME Ltd", "accounts": 3, "sanctions_hit": False, "pep": False,
               "turnover": 4_200_000, "arrears": 0},
    "C-9001": {"name": "Vulpes SA", "accounts": 1, "sanctions_hit": True, "pep": True,
               "turnover": 90_000, "arrears": 2},
}

def data_retriever(state: SupervisorState) -> dict:
    """Only node with access to the customer store."""
    cid = state["customer_id"]
    if cid not in CUSTOMERS:
        msg = AgentMessage(sender=Specialist.RETRIEVER, ok=False,
                           summary=f"no record for {cid}", error="CustomerNotFound")
        return {"messages": [msg.model_dump(mode="json")], "visited": ["data_retriever"],
                "failures": ["data_retriever"]}
    rec = CUSTOMERS[cid]
    msg = AgentMessage(sender=Specialist.RETRIEVER,
                       summary=f"retrieved {rec['name']}: {rec['accounts']} accounts",
                       data=rec)
    return {"messages": [msg.model_dump(mode="json")], "visited": ["data_retriever"]}

def compliance_checker(state: SupervisorState) -> dict:
    """Reads what the retriever found; cannot write the report."""
    rec = next((m["data"] for m in reversed(state["messages"])
                if m["sender"] == Specialist.RETRIEVER and m["ok"]), None)
    if rec is None:
        msg = AgentMessage(sender=Specialist.COMPLIANCE, ok=False,
                           summary="no customer data to check", error="MissingUpstream")
        return {"messages": [msg.model_dump(mode="json")], "visited": ["compliance_checker"],
                "failures": ["compliance_checker"]}
    flags = [f for f, hit in [("sanctions", rec["sanctions_hit"]), ("pep", rec["pep"]),
                              ("arrears", rec["arrears"] > 0)] if hit]
    verdict = "REFER" if flags else "CLEAR"
    msg = AgentMessage(sender=Specialist.COMPLIANCE,
                       summary=f"verdict {verdict}" + (f" (flags: {', '.join(flags)})" if flags else ""),
                       data={"verdict": verdict, "flags": flags})
    return {"messages": [msg.model_dump(mode="json")], "visited": ["compliance_checker"]}

def report_generator(state: SupervisorState) -> dict:
    """Composes the final answer from every upstream message."""
    lines = [f"REPORT — {state['request']}", f"customer: {state['customer_id']}"]
    lines += [f"  [{m['sender']}] {m['summary']}" for m in state["messages"]]
    if state["failures"]:
        lines.append(f"  ! degraded: {', '.join(sorted(set(state['failures'])))} unavailable")
    report = "\n".join(lines)
    msg = AgentMessage(sender=Specialist.REPORTER, summary="report written",
                       data={"chars": len(report)})
    return {"messages": [msg.model_dump(mode="json")], "visited": ["report_generator"], "report": report}

print("3 specialists defined — each returns a typed AgentMessage")

3 specialists defined — each returns a typed AgentMessage


☝ The boundary is structural, not documentary: `compliance_checker` has no access to `CUSTOMERS` and physically cannot produce a report; the retriever cannot render one. An auditor can verify that by reading the function bodies — which is worth far more than a paragraph in a design doc claiming separation of duties.

Also note every specialist handles its own missing-upstream case and returns `ok=False` rather than raising. A specialist's job is to *report* failure in the contract's shape; deciding what to do about it is the supervisor's job.

## 3. The supervisor + conditional routing

The supervisor is a node that returns *which node runs next*. `add_conditional_edges` turns that string into an actual edge. The loop is: supervisor → specialist → supervisor → … → END.

In [3]:
from langgraph.graph import StateGraph, START, END

def supervisor(state: SupervisorState) -> dict:
    """Decides the next specialist from what's already been done.

    Rule-based here so the notebook is deterministic; swap the body for a Bedrock
    call returning a Specialist and the graph below does not change.
    """
    visited = state["visited"]
    if Specialist.RETRIEVER not in visited:
        nxt = Specialist.RETRIEVER
    elif Specialist.COMPLIANCE not in visited:
        nxt = Specialist.COMPLIANCE
    elif Specialist.REPORTER not in visited:
        nxt = Specialist.REPORTER
    else:
        nxt = Specialist.DONE
    return {"next": nxt.value, "messages": [{"sender": "supervisor", "ok": True,
                                       "summary": f"route -> {nxt}", "data": {}, "error": None}]}

def route(state: SupervisorState) -> str:
    return END if state["next"] == Specialist.DONE else state["next"]

def build_graph():
    g = StateGraph(SupervisorState)
    g.add_node("supervisor", supervisor)
    g.add_node(Specialist.RETRIEVER.value, data_retriever)
    g.add_node(Specialist.COMPLIANCE.value, compliance_checker)
    g.add_node(Specialist.REPORTER.value, report_generator)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", route,
                            {Specialist.RETRIEVER: Specialist.RETRIEVER,
                             Specialist.COMPLIANCE: Specialist.COMPLIANCE,
                             Specialist.REPORTER: Specialist.REPORTER,
                             END: END})
    for s in (Specialist.RETRIEVER, Specialist.COMPLIANCE, Specialist.REPORTER):
        g.add_edge(s.value, "supervisor")                 # every specialist reports back
    return g.compile()

graph = build_graph()
print(graph.get_graph().draw_ascii())

                                                +-----------+                                             
                                                | __start__ |                                             
                                                +-----------+                                             
                                                       *                                                  
                                                       *                                                  
                                                       *                                                  
                                                +------------+                                            
                                             ...| supervisor |.....                                       
                                     ........   +------------+.    .......                                
                              .......

In [4]:
INIT = {"messages": [], "visited": [], "failures": [], "next": "", "report": ""}

out = graph.invoke({**INIT, "request": "Onboarding review", "customer_id": "C-4417"})
print("HOPS:")
for m in out["messages"]:
    print(f"  {m['sender']:<20} {m['summary']}")
print("\n" + out["report"])

HOPS:
  supervisor           route -> data_retriever
  data_retriever       retrieved ACME Ltd: 3 accounts
  supervisor           route -> compliance_checker
  compliance_checker   verdict CLEAR
  supervisor           route -> report_generator
  report_generator     report written
  supervisor           route -> __done__

REPORT — Onboarding review
customer: C-4417
  [supervisor] route -> data_retriever
  [data_retriever] retrieved ACME Ltd: 3 accounts
  [supervisor] route -> compliance_checker
  [compliance_checker] verdict CLEAR
  [supervisor] route -> report_generator


☝ Every specialist returns to the supervisor — that's the "all replies return to the router" shape from Day 31, and it's what gives you **one** place that sees every result, logs it, and decides what happens next. The `messages` list is the audit trail, produced for free by the reducer.

The important seam: `supervisor()` is rule-based, so this notebook is deterministic and testable. Replace the body with an LLM call that returns a `Specialist` and *nothing else changes* — same graph, same edges, same tests. Keep routing logic behind that seam and you can unit-test your topology without spending a token.

## 4. Failure and fallback routing

A specialist will fail — a customer isn't found, a KB times out. The supervisor's real job is deciding between **retry**, **route around**, and **degrade**.

In [5]:
MAX_ATTEMPTS = 2

class ResilientState(SupervisorState):
    attempts: Annotated[list[str], operator.add]

def resilient_supervisor(state: ResilientState) -> dict:
    visited, failures = state["visited"], state["failures"]
    attempts = state["attempts"]

    def failed(node: str) -> bool:
        return node in failures

    def tries(node: str) -> int:
        return attempts.count(node)

    # 1. retry a failed specialist, up to MAX_ATTEMPTS
    for node in (Specialist.RETRIEVER, Specialist.COMPLIANCE):
        if failed(node) and tries(node) < MAX_ATTEMPTS:
            return {"next": node.value, "attempts": [node.value],
                    "messages": [{"sender": "supervisor", "ok": True,
                                  "summary": f"retry {node} (attempt {tries(node)+1})",
                                  "data": {}, "error": None}]}
    # 2. route around: retriever dead -> skip compliance (it has no data to check)
    if failed(Specialist.RETRIEVER) and Specialist.COMPLIANCE not in visited:
        return {"next": Specialist.REPORTER.value, "visited": [Specialist.COMPLIANCE.value],
                "messages": [{"sender": "supervisor", "ok": True,
                              "summary": "retriever exhausted -> skip compliance, degrade to report",
                              "data": {}, "error": None}]}
    # 3. normal order
    for node in (Specialist.RETRIEVER, Specialist.COMPLIANCE, Specialist.REPORTER):
        if node not in visited:
            return {"next": node.value, "attempts": [node.value],
                    "messages": [{"sender": "supervisor", "ok": True,
                                  "summary": f"route -> {node}", "data": {}, "error": None}]}
    return {"next": Specialist.DONE.value,
            "messages": [{"sender": "supervisor", "ok": True, "summary": "route -> done",
                          "data": {}, "error": None}]}

def build_resilient():
    g = StateGraph(ResilientState)
    g.add_node("supervisor", resilient_supervisor)
    g.add_node(Specialist.RETRIEVER.value, data_retriever)
    g.add_node(Specialist.COMPLIANCE.value, compliance_checker)
    g.add_node(Specialist.REPORTER.value, report_generator)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", route,
                            {Specialist.RETRIEVER: Specialist.RETRIEVER,
                             Specialist.COMPLIANCE: Specialist.COMPLIANCE,
                             Specialist.REPORTER: Specialist.REPORTER, END: END})
    for s in (Specialist.RETRIEVER, Specialist.COMPLIANCE, Specialist.REPORTER):
        g.add_edge(s.value, "supervisor")
    return g.compile()

rgraph = build_resilient()
bad = rgraph.invoke({**INIT, "attempts": [], "request": "Onboarding review",
                     "customer_id": "C-0000"})          # unknown customer -> retriever fails
print("SUPERVISOR DECISIONS (failing retriever):")
for m in bad["messages"]:
    mark = " " if m["ok"] else "!"
    print(f" {mark} {m['sender']:<20} {m['summary']}")
print("\n" + bad["report"])

SUPERVISOR DECISIONS (failing retriever):
   supervisor           route -> data_retriever
 ! data_retriever       no record for C-0000
   supervisor           retry data_retriever (attempt 2)
 ! data_retriever       no record for C-0000
   supervisor           retriever exhausted -> skip compliance, degrade to report
   report_generator     report written
   supervisor           route -> done

REPORT — Onboarding review
customer: C-0000
  [supervisor] route -> data_retriever
  [data_retriever] no record for C-0000
  [supervisor] retry data_retriever (attempt 2)
  [data_retriever] no record for C-0000
  [supervisor] retriever exhausted -> skip compliance, degrade to report
  ! degraded: data_retriever unavailable


☝ The request **completed** despite its first agent failing twice: attempt → retry → route around → a degraded report that says so. Three properties worth naming in a design review:

- **The attempt cap is what makes the loop terminate.** Without `MAX_ATTEMPTS`, an LLM supervisor watching a failure will keep re-routing to it forever, and you'll pay for every hop. `recursion_limit` on `.invoke()` is your backstop, not your plan.
- **Route-around beats crash.** Compliance had nothing to check, so the supervisor marked it visited and moved on rather than letting it fail too.
- **Degradation is stated, not hidden.** `! degraded: data_retriever unavailable` appears in the report. Silently returning a confident answer built on missing data is the failure mode that ends careers in banking — say what's missing.

## 5. Checkpointing + the human gate

Two more things production needs: **resume after a crash**, and a **human approving** anything consequential. Both are LangGraph features you met in Phase 1 — here they land on the supervisor.

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

def build_gated():
    g = StateGraph(SupervisorState)
    g.add_node("supervisor", supervisor)
    g.add_node(Specialist.RETRIEVER.value, data_retriever)
    g.add_node(Specialist.COMPLIANCE.value, compliance_checker)
    g.add_node(Specialist.REPORTER.value, report_generator)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", route,
                            {Specialist.RETRIEVER: Specialist.RETRIEVER,
                             Specialist.COMPLIANCE: Specialist.COMPLIANCE,
                             Specialist.REPORTER: Specialist.REPORTER, END: END})
    for s in (Specialist.RETRIEVER, Specialist.COMPLIANCE, Specialist.REPORTER):
        g.add_edge(s.value, "supervisor")
    # pause before the report whenever compliance has referred the case
    return g.compile(checkpointer=InMemorySaver(), interrupt_before=[Specialist.REPORTER.value])

gated = build_gated()
cfg = {"configurable": {"thread_id": "case-9001"}}

paused = gated.invoke({**INIT, "request": "Onboarding review", "customer_id": "C-9001"}, cfg)
verdict = next(m for m in reversed(paused["messages"]) if m["sender"] == Specialist.COMPLIANCE)
print("PAUSED before:", gated.get_state(cfg).next)
print("compliance   :", verdict["summary"])
print("-> human reviews the sanctions/PEP hit here; the graph is durable on the checkpointer\n")

resumed = gated.invoke(None, cfg)                    # human approved -> continue
print(resumed["report"])
print("\nthread state after resume:", gated.get_state(cfg).next or "(complete)")

PAUSED before: ('report_generator',)
compliance   : verdict REFER (flags: sanctions, pep, arrears)
-> human reviews the sanctions/PEP hit here; the graph is durable on the checkpointer

REPORT — Onboarding review
customer: C-9001
  [supervisor] route -> data_retriever
  [data_retriever] retrieved Vulpes SA: 1 accounts
  [supervisor] route -> compliance_checker
  [compliance_checker] verdict REFER (flags: sanctions, pep, arrears)
  [supervisor] route -> report_generator

thread state after resume: (complete)


☝ `interrupt_before` + a checkpointer is the whole human-in-the-loop mechanism: the graph **stops**, its state is durable under `thread_id="case-9001"`, and `invoke(None, cfg)` resumes exactly where it paused — hours later, in a different process, after a deploy. Swap `InMemorySaver` for `SqliteSaver`/`PostgresSaver` and it survives a restart.

This is the concrete reason a regulated flow goes on LangGraph rather than a crew: a sanctions hit **must** reach a human, that pause must survive infrastructure, and the resume must be auditable. `Process.sequential` has nowhere to put that.

## 6. Recap — the supervisor is the production topology

| Requirement | Mechanism | Cell |
|---|---|---|
| Agents can't send junk to each other | Pydantic `AgentMessage` + `StrEnum` routing keys | 1 |
| Separation of duties | only the retriever reads customer data | 2 |
| Runtime routing | `add_conditional_edges` on `supervisor` | 3 |
| Full audit trail | `Annotated[list, operator.add]` reducer | 3 |
| Specialist fails | retry → route-around → **stated** degradation | 4 |
| Loop can't run away | attempt cap (+ `recursion_limit` backstop) | 4 |
| Human gate | `interrupt_before=[...]` + checkpointer | 5 |
| Survive a crash | `thread_id` + `SqliteSaver`/`PostgresSaver` | 5 |

The cost is real — supervisor + 3 specialists is ~7 LLM calls per request where a single tool-using agent is 1 (Day 31's arithmetic). You pay it for the things in that table, and you should be able to say which line justifies it. Here: the sanctions gate and the access boundary do.

**CrewAI vs LangGraph, from having built both:** CrewAI when it's a chain of role-shaped steps and you want it running this afternoon. LangGraph when routing is dynamic, failure needs a fallback, a human must approve, or the run must resume — i.e. anything a Barclays risk forum will review.

Tomorrow: **AutoGen** and Semantic Kernel — the third framework, and the comparison that completes the picture.